In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image

In [2]:
# Install required libraries for Qwen3-VL-2B-Instruct
!pip install -q torchvision accelerate transformers>=4.45.0

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

QWEN_MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"

# Initialize processor and model
processor_qwen = AutoProcessor.from_pretrained(QWEN_MODEL_NAME)

model_qwen = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_qwen.eval()

print("Qwen3-VL-2B-Instruct loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL-2B-Instruct loaded successfully!


In [10]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_path"] == target_name:
            return record["question"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [5]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [6]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [7]:

def evaluate(query_text, b64_str):
    # Decode the base64 string to a PIL Image
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_qwen(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model_qwen.device)

    with torch.inference_mode():
        output_ids = model_qwen.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_qwen.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()

In [12]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- Paths Configuration (Adjust as needed) ---
IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/annotations.json"

# Load annotations lookup map for faster access
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Create a lookup map for ground truth: (image_id) -> label
# Since we are evaluating the actual files in /images/, we target '/images/' records
ground_truth_map = {}
for record in annotations_list:
    if record["image_path"].startswith("images/"):
        img_id = record["image_id"]
        ground_truth_map[int(img_id)] = record["label"]

# Get list of all images in the directory
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

results = []

print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    # Extract image_id from filename (e.g., '000000046207.jpg' -> 46207)
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under all conditions
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"
         
        
        
    results.append({
        "Image_Id": image_id,
        "Ground_truth":ground_truth,
        "Original": orig_val})

df_results = pd.DataFrame(results)
display(df_results.head(10))

Processing 491 images through all variations...


  0%|          | 0/491 [00:00<?, ?it/s]

,Image_Id,Ground_truth,Original
0,102439,yes,Yes
1,236865,yes,Yes
2,327038,yes,Yes
3,381895,yes,Yes
4,108189,yes,Yes
5,153865,yes,Yes
6,560744,yes,Yes
7,408757,yes,Yes
8,562155,yes,Yes
9,579231,yes,Yes


In [13]:
df_results.rename(columns={"Original": "Qwen-2B"}, inplace=True)
df_results

,Image_Id,Ground_truth,Qwen-2B
0,102439,yes,Yes
1,236865,yes,Yes
2,327038,yes,Yes
3,381895,yes,Yes
4,108189,yes,Yes
...,...,...,...
486,172648,yes,Yes
487,365317,yes,Yes
488,573796,yes,Yes
489,528136,yes,Yes


In [17]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/diff-models-evals-on-pope/diff_models_eval (1).csv", usecols=["Image_Id","Ground_truth","Qwen-2B"])
df

,Image_Id,Ground_truth,Qwen-2B
0,6578,No,No
1,8602,No,No
2,9469,No,No
3,9712,No,Yes
4,10733,No,No
...,...,...,...
495,574356,No,No
496,574837,No,No
497,575121,No,No
498,575916,No,No


In [22]:
df2 = pd.concat([df,df_results], ignore_index=True)
df2

,Image_Id,Ground_truth,Qwen-2B
0,6578,No,No
1,8602,No,No
2,9469,No,No
3,9712,No,Yes
4,10733,No,No
...,...,...,...
986,172648,yes,Yes
987,365317,yes,Yes
988,573796,yes,Yes
989,528136,yes,Yes


In [23]:
df2 = df2.drop_duplicates(subset=["Image_Id"], keep="first")
df2

,Image_Id,Ground_truth,Qwen-2B
0,6578,No,No
1,8602,No,No
2,9469,No,No
3,9712,No,Yes
4,10733,No,No
...,...,...,...
986,172648,yes,Yes
987,365317,yes,Yes
988,573796,yes,Yes
989,528136,yes,Yes


In [25]:
df3=pd.read_csv("/kaggle/input/datasets/ramakanthjogada/mnbvcz/two-model.csv")
df3

,Image_Id,Ground_truth,Qwen-3B,Smol-2.2B
0,169361,No,No,No.
1,47022,No,No,Yes.
2,556064,No,No,No.
3,474004,No,No,Yes.
4,306710,No,No,Yes.
...,...,...,...,...
984,497599,yes,Yes.,Yes.
985,121959,yes,Yes,Yes.
986,361430,yes,No,No.
987,277289,yes,Yes,Yes.


In [26]:

merged_df = pd.merge(df2, df3, on="Image_Id", how="inner")
merged_df.reset_index(drop=True, inplace=True)

In [27]:
merged_df

,Image_Id,Ground_truth_x,Qwen-2B,Ground_truth_y,Qwen-3B,Smol-2.2B
0,6578,No,No,No,Yes.,Yes.
1,8602,No,No,No,No,No.
2,9469,No,No,No,No,No.
3,9712,No,Yes,No,Yes.,No.
4,10733,No,No,No,No,No.
...,...,...,...,...,...,...
984,172648,yes,Yes,yes,Yes.,Yes.
985,365317,yes,Yes,yes,Yes,Yes.
986,573796,yes,Yes,yes,Yes.,Yes.
987,528136,yes,Yes,yes,Yes.,Yes.


In [28]:
merged_df.drop(columns=["Ground_truth_y"], inplace=True)
merged_df

,Image_Id,Ground_truth_x,Qwen-2B,Qwen-3B,Smol-2.2B
0,6578,No,No,Yes.,Yes.
1,8602,No,No,No,No.
2,9469,No,No,No,No.
3,9712,No,Yes,Yes.,No.
4,10733,No,No,No,No.
...,...,...,...,...,...
984,172648,yes,Yes,Yes.,Yes.
985,365317,yes,Yes,Yes,Yes.
986,573796,yes,Yes,Yes.,Yes.
987,528136,yes,Yes,Yes.,Yes.


In [30]:
merged_df.rename(columns={"Ground_truth_x": "Ground_truth"}, inplace=True)
merged_df

,Image_Id,Ground_truth,Qwen-2B,Qwen-3B,Smol-2.2B
0,6578,No,No,Yes.,Yes.
1,8602,No,No,No,No.
2,9469,No,No,No,No.
3,9712,No,Yes,Yes.,No.
4,10733,No,No,No,No.
...,...,...,...,...,...
984,172648,yes,Yes,Yes.,Yes.
985,365317,yes,Yes,Yes,Yes.
986,573796,yes,Yes,Yes.,Yes.
987,528136,yes,Yes,Yes.,Yes.


In [32]:
merged_df.to_csv("diff-models-evals-on-POPE-final.csv")